In [1]:
import sys, re, time
sys.path.append('../src')
import pandas as pd
from data import load_split
from features import (url_length, count_dots, count_hyphens, count_digits,
                      count_at_symbols, has_ip_address, has_https,
                      count_subdomains, has_suspicious_words)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

print("Outils importés avec succès !")

Outils importés avec succès !


In [2]:
# Dictionnaire qui associe une colonne à sa fonction de calcul
FEATURES = {
    'longueur': url_length, 'nb_points': count_dots, 'nb_tirets': count_hyphens,
    'nb_chiffres': count_digits, 'nb_arobase': count_at_symbols,
    'a_ip': has_ip_address, 'a_https': has_https,
    'nb_sous_domaines': count_subdomains, 'mots_suspects': has_suspicious_words,
}
cols = list(FEATURES)

# Fonction pour enlever http:// et www.
def nettoyer(u):
    u = re.sub(r"^https?://", "", str(u))
    return re.sub(r"^www\.", "", u)

print("Fonctions prêtes !")

Fonctions prêtes !


In [3]:
# Découpage propre (GroupShuffleSplit + sans doublons)
print("Chargement et découpage des données...")
df_train_full, df_test_full = load_split()

# On prend un échantillon pour aller vite
df_train = df_train_full.sample(n=40000, random_state=42).copy()
df_test = df_test_full.sample(n=10000, random_state=42).copy()

y_train = df_train['status']
y_test = df_test['status']

print(f"Train : {len(df_train)} lignes | Test : {len(df_test)} lignes")

Chargement et découpage des données...
Train : 40000 lignes | Test : 10000 lignes


In [5]:
print("Calcul des features numériques...")
for col, f in FEATURES.items():
    df_train[col] = df_train['url'].apply(f)
    df_test[col] = df_test['url'].apply(f)

# Nettoyage des URL (sans http:// et www.)
df_train['url_propre'] = df_train['url'].apply(nettoyer)
df_test['url_propre'] = df_test['url'].apply(nettoyer)

print("Features et URL propres calculées !")

Calcul des features numériques...
Features et URL propres calculées !


In [6]:
print("Création des n-grammes...")
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=2000)
Xtr_ng = vectorizer.fit_transform(df_train['url_propre'])
Xte_ng = vectorizer.transform(df_test['url_propre'])

print(f"Matrice d'entraînement : {Xtr_ng.shape}")
print(f"Matrice de test : {Xte_ng.shape}")

Création des n-grammes...
Matrice d'entraînement : (40000, 2000)
Matrice de test : (10000, 2000)


In [7]:
resultats = []

def evaluer(nom, modele, Xtr, Xte):
    print(f"Entraînement : {nom}...")
    t = time.time()
    modele.fit(Xtr, y_train)
    duree = time.time() - t
    pred = modele.predict(Xte)
    
    resultats.append({
        'Modèle': nom,
        'Exactitude': accuracy_score(y_test, pred),
        'Précision (phishing)': precision_score(y_test, pred, pos_label=0),
        'Rappel (phishing)': recall_score(y_test, pred, pos_label=0),
        'F1 (phishing)': f1_score(y_test, pred, pos_label=0),
        'Temps (s)': duree,
    })
    print(f"  -> Terminé en {duree:.2f}s")

print("Fonction d'évaluation prête !")

Fonction d'évaluation prête !


In [8]:
# Test 1 : Random Forest sur les chiffres
evaluer('RF - chiffres', 
        RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42),
        df_train[cols], df_test[cols])

# Test 2 : XGBoost sur les chiffres
evaluer('XGB - chiffres', 
        XGBClassifier(n_estimators=50, n_jobs=-1, tree_method='hist', random_state=42),
        df_train[cols], df_test[cols])

# Test 3 : Random Forest sur les n-grammes
evaluer('RF - n-grammes', 
        RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42),
        Xtr_ng, Xte_ng)

# Test 4 : XGBoost sur les n-grammes
evaluer('XGB - n-grammes', 
        XGBClassifier(n_estimators=50, n_jobs=-1, tree_method='hist', random_state=42),
        Xtr_ng, Xte_ng)

print("\nTous les modèles ont été entraînés !")

Entraînement : RF - chiffres...
  -> Terminé en 0.69s
Entraînement : XGB - chiffres...
  -> Terminé en 0.39s
Entraînement : RF - n-grammes...
  -> Terminé en 13.27s
Entraînement : XGB - n-grammes...
  -> Terminé en 18.26s

Tous les modèles ont été entraînés !


In [9]:
tableau = pd.DataFrame(resultats).round(3)
tableau

,Modèle,Exactitude,Précision (phishing),Rappel (phishing),F1 (phishing),Temps (s)
0,RF - chiffres,0.783,0.767,0.757,0.762,0.690
1,XGB - chiffres,0.813,0.833,0.740,0.784,0.391
2,RF - n-grammes,0.899,0.901,0.877,0.889,13.267
3,XGB - n-grammes,0.899,0.907,0.867,0.887,18.258


In [11]:
import joblib
from sklearn.ensemble import RandomForestClassifier

print("Recréation et entraînement du modèle Random Forest (n-grammes)...")

# 1. On recrée le modèle champion
rf_ngram_final = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42)

# 2. On l'entraîne (il utilisera les données Xtr_ng et y_train déjà en mémoire)
rf_ngram_final.fit(Xtr_ng, y_train)

print("Entraînement terminé !")

# 3. On sauvegarde le modèle ET le vectorizer
joblib.dump(rf_ngram_final, '../src/model_final_rf.pkl')
joblib.dump(vectorizer, '../src/vectorizer_final.pkl')

print("Modèle final et vectorizer sauvegardés avec succès dans le dossier src/ !")

Recréation et entraînement du modèle Random Forest (n-grammes)...
Entraînement terminé !
Modèle final et vectorizer sauvegardés avec succès dans le dossier src/ !
